In [ ]:
pip install pystac-client odc-stac rioxarray dask['distributed'] jupyter-server-proxy xrscipy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.6/159.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.4 MB/s eta 0:00:00


In [ ]:
import dask
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import os
import pyproj
import pystac_client
import rioxarray as rxr
import xarray as xr
import xrscipy.signal as xrs
from odc.stac import configure_s3_access, load

In [ ]:
from dask.distributed import Client
client = Client() # set up local cluster on the machine
client

INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:36213
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:45559'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:44739'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:37007 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:37007
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:38092
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:39277 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:39277
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:38098
INFO:distributed.scheduler:Receive client connection: Client-39a14696-b5d1-11f1-81f3-0242ac1c000c
INFO:distributed.core:Starting establish

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 2,Total memory: 12.67 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36213,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:37007,Total threads: 1
Dashboard: http://127.0.0.1:42487/status,Memory: 6.34 GiB
Nanny: tcp://127.0.0.1:45559,


### Import vector Embalse Sa Roque

In [ ]:
# --
import geopandas as gpd
esr_vector = gpd.read_file('https://github.com/francobarrionuevoenv21/MAIE_devs_pub/raw/refs/heads/main/Proc_Img/tp_final_claS2/data/esr_clean.geojson').to_crs('EPSG:4326')
esr_vector = esr_vector.dissolve()

# --
minx, miny, maxx, maxy = esr_vector.total_bounds

In [ ]:
minx, miny, maxx, maxy

(np.float64(-64.49914797926462),
 np.float64(-31.415812782729862),
 np.float64(-64.4325891780906),
 np.float64(-31.31939141738306))

### Configuración search & request desde STAC

In [ ]:
# --
# Se define desde 2023 hasta 2025, inclusive, ya que
date_ini = '2023-01-31'
date_end= '2026-01-01'


In [ ]:
# Query the STAC Catalog
catalog = pystac_client.Client.open(
    'https://earth-search.aws.element84.com/v1')

search = catalog.search(
    collections=['sentinel-2-c1-l2a'], #'sentinel-2-l2a'
    bbox= [minx, miny, maxx, maxy],
    datetime=f'{date_ini}/{date_end}',
    sortby=[{"field": "properties.datetime", "direction": "asc"}],  # oldest first
    query={
        'eo:cloud_cover': {'lt': 20},
    }
)
items = search.item_collection()

# Load to XArray
ds = load(
    items,
    bands=['red', 'nir', 'scl'], # bands=['red', 'green', 'blue', 'nir', 'scl'],
    bbox=[minx, miny, maxx, maxy],      # your AOI (lon/lat)
    resolution=10,
    crs='utm',
    #chunks={'x': 4096, 'y': 4096, 'time': 1}, #chunks={'x': 1024, 'y': 1024},  # Explicitly define chunk sizes
    chunks={'time': 1, 'x': -1, 'y': -1},   # one spatial chunk per date: matches how the data is read
    # In Dask and xarray, a chunk size of -1 means "use the full length of this dimension as a single chunk."
    groupby='solar_day',
    preserve_original_order=True
)
#ds

/usr/local/lib/python3.13/dist-packages/pystac/extensions/storage.py:724: UserWarning: Could not parse bucket/account from href. The following assets were not migrated: ['red', 'green', 'blue', 'visual', 'nir', 'swir22', 'rededge2', 'rededge3', 'rededge1', 'swir16', 'wvp', 'nir08', 'scl', 'aot', 'coastal', 'nir09', 'cloud', 'snow', 'preview', 'granule_metadata', 'tileinfo_metadata', 'product_metadata', 'thumbnail']
  warnings.warn(


In [ ]:
# --
ds

<xarray.Dataset> Size: 678MB
Dimensions:      (y: 1078, x: 648, time: 194)
Coordinates:
  * y            (y) float64 9kB 6.534e+06 6.534e+06 ... 6.523e+06 6.523e+06
  * x            (x) float64 5kB 3.574e+05 3.574e+05 ... 3.638e+05 3.638e+05
  * time         (time) datetime64[ns] 2kB 2023-02-06T14:31:21.521000 ... 202...
    spatial_ref  int32 4B 32720
Data variables:
    red          (time, y, x) uint16 271MB dask.array<chunksize=(1, 1078, 648), meta=np.ndarray>
    nir          (time, y, x) uint16 271MB dask.array<chunksize=(1, 1078, 648), meta=np.ndarray>
    scl          (time, y, x) uint8 136MB dask.array<chunksize=(1, 1078, 648), meta=np.ndarray>

In [ ]:
# Apply scale/offset
scale = 0.0001
offset = -0.1
# Select spectral bands (all except 'scl')
data_bands = ['red', 'nir']
for band in data_bands:
  ds[band] = ds[band] * scale + offset

In [ ]:
# Computo la concentración de cl-a según modelo semiempírico de German et al. (2021)
da_chla = -5.57 + 80.13 * (ds['nir'] / ds['red'])
da_chla = da_chla.clip(2.8, 288.5)  # Defino límites acorde a valores del modelo

In [ ]:
# --
ds_chla = xr.Dataset(data_vars={'chla': da_chla, 'slc': ds['scl']})

In [ ]:
# --
esr_reprj = esr_vector.to_crs('EPSG:32720')
ds_chla_clip = ds_chla.rio.clip(esr_reprj.geometry)

**Preview scene**

In [ ]:
# --
scene_prevw = ds_chla_clip.isel(time=0)['chla']

#scene = ds.squeeze()

In [ ]:
'''preview = scene_prevw.rio.reproject(
    scene_prevw.rio.crs, resolution=300
)
preview'''

'preview = scene_prevw.rio.reproject(\n    scene_prevw.rio.crs, resolution=300\n)\npreview'

In [ ]:
'''
Let’s call compute() to kick-off the dask graph. Dask will query the cloud-hosted dataset to fetch the required pixels. Once you run the cell, look at the Dask Diagnostic Dashboard to see the data processing in action.
'''
'''%%time
scene = scene_prevw.compute()'''

'%%time\nscene = scene_prevw.compute()'

In [ ]:
import matplotlib.pyplot as plt

red = ds_chla_clip['chla']#ds['red']

# Mask nodata (0) before averaging, otherwise it drags the mean down
red = red.where(red != 0)

ts = red.mean(dim=['x', 'y'], skipna=True).compute()

ts.plot(marker='o', linestyle='-', figsize=(12, 4))
plt.title('Red band: spatial mean per date')
plt.ylabel('Digital number / reflectance')
plt.show()

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-595056' coro=<Client._gather.<locals>.wait() done, defined at /usr/local/lib/python3.13/dist-packages/distributed/client.py:2367> exception=AllExit()>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/distributed/client.py", line 2376, in wait
    raise AllExit()
distributed.client.AllExit


KeyboardInterrupt: 

In [ ]:
ds_chla_clip.to_netcdf('chla_slc_clip_2326.nc')